# Task 08: Transformer for IceCube position reconstruction

This notebook is built step by step. The goal is to train a Transformer encoder that reconstructs the two-dimensional neutrino interaction position from variable-length IceCube hit sequences.

Each event contains a sequence of detector hits with time and detector-position information. Since different events have different numbers of hits, the notebook will explicitly handle padding, masking, and masked pooling before predicting the target coordinates `[xpos, ypos]`.

### 1. Imports and setup

This first section imports the tools needed later, configures plotting, fixes random seeds, and selects the available PyTorch device.

In [ ]:
from pathlib import Path
import random

from common.setup_plotting import setup_matplotlib, get_figure_dir
from common.training import save_checkpoint, load_checkpoint

import awkward as ak
import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

setup_matplotlib()        # configure matplotlib first (So i can use LaTeX in the labels)
# Make interactive plots work in Jupyter notebooks
%matplotlib inline
import matplotlib.pyplot as plt   # THEN import pyplot

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

fig_dir = get_figure_dir("task_08")
data_dir = Path("../../../data/task_08/IceCube 2D Dataset")
checkpoint_path = Path("../models/transformer_encoder.pt")

plot_label_size = 13
plot_title_size = 14
plot_tick_size = 11
plot_legend_size = 11

print("device:", device)
print("data directory:", data_dir)
print("figure directory:", fig_dir)
print("checkpoint path:", checkpoint_path)

### 2. Hyperparameters

The next cell defines the training and Transformer settings. The model uses at least two attention heads and two encoder layers, as required by the task. The hidden dimension is kept modest so the notebook remains practical to train while still showing the full Transformer workflow.

In [ ]:
batch_size = 64
num_epochs = 50
learning_rate = 1e-3
weight_decay = 1e-4
patience = 6

input_dim = 3          # hit features: time, detector x, detector y
output_dim = 2         # target coordinates: xpos, ypos
hidden_dim = 64
num_attention_heads = 4
num_encoder_layers = 2
dim_feedforward = 128
dropout = 0.05

skip_training = False

### 3. Load the IceCube datasets

The IceCube data is stored in three parquet files: one split for training, one for validation, and one for final testing. We keep these splits separate throughout the notebook so the test set is only used after model development is finished.

In [ ]:
train_dataset = ak.from_parquet(data_dir / "train.pq")
val_dataset = ak.from_parquet(data_dir / "val.pq")
test_dataset = ak.from_parquet(data_dir / "test.pq")

print("Dataset sizes (train, val, test):", len(train_dataset), len(val_dataset), len(test_dataset))
print("Units x position, y position, time: m, m, ns")

### 4. Inspect one event

Before building the model, we first inspect the data layout. Each event stores its measured hit sequence in the `data` field and its target interaction position in `xpos` and `ypos`.

In [ ]:
print(f"The training dataset contains {len(train_dataset)} events.")
print(f"The validation dataset contains {len(val_dataset)} events.")
print(f"The test dataset contains {len(test_dataset)} events.")
print("Training dataset columns:", train_dataset.fields)

first_event = train_dataset[0]
first_event_data = first_event["data"]
first_event_hits = len(first_event_data[0])

print("\nFirst event:")
print(first_event)
print("\nFirst event target [xpos, ypos]:", first_event["xpos"], first_event["ypos"])
print("First event data shape [features, hits]:", ak.to_numpy(ak.num(first_event_data, axis=1)))
print("Number of hits in first event:", first_event_hits)

print("\nFirst few hits of the first event:")
for hit_idx in range(min(first_event_hits, 5)):
    hit_time = first_event_data[0, hit_idx]
    hit_x = first_event_data[1, hit_idx]
    hit_y = first_event_data[2, hit_idx]
    print(f"Hit {hit_idx}: time = {hit_time:.3f}, x = {hit_x:.3f}, y = {hit_y:.3f}")

### 5. Dataset overview plots

The hit count tells us how much padding a batch may need, while the target-coordinate distributions show the physical range of the regression problem.

In [ ]:
train_hit_counts = ak.to_numpy(ak.num(train_dataset["data"][:, 0, :], axis=1))
val_hit_counts = ak.to_numpy(ak.num(val_dataset["data"][:, 0, :], axis=1))
test_hit_counts = ak.to_numpy(ak.num(test_dataset["data"][:, 0, :], axis=1))

train_xpos = ak.to_numpy(train_dataset["xpos"])
train_ypos = ak.to_numpy(train_dataset["ypos"])
val_xpos = ak.to_numpy(val_dataset["xpos"])
test_xpos = ak.to_numpy(test_dataset["xpos"])

print("Data files:")
print("train:", data_dir / "train.pq")
print("validation:", data_dir / "val.pq")
print("test:", data_dir / "test.pq")
print("\nDataset sizes (train, val, test):", len(train_dataset), len(val_dataset), len(test_dataset))
print("First xpos values (train, val, test):", train_xpos[0], val_xpos[0], test_xpos[0])
print("\nTraining hit-count summary:")
print("min:", train_hit_counts.min())
print("median:", np.median(train_hit_counts))
print("mean:", train_hit_counts.mean())
print("max:", train_hit_counts.max())

plt.close("all")
fig, axes = plt.subplots(2, 2, figsize=(8.6, 7.4))
axes = axes.ravel()

axes[0].hist(train_hit_counts, bins=50, alpha=0.75, color="C0", label="train")
axes[0].hist(val_hit_counts, bins=50, alpha=0.45, color="C1", label="validation")
axes[0].hist(test_hit_counts, bins=50, alpha=0.35, color="C2", label="test")
axes[0].set_xlabel("Number of hits", fontsize=plot_label_size)
axes[0].set_ylabel("Number of events", fontsize=plot_label_size)
axes[0].set_title("Hit multiplicity", fontsize=plot_title_size)
axes[0].tick_params(labelsize=plot_tick_size)
axes[0].legend(fontsize=plot_legend_size)

axes[1].hist(train_xpos, bins=50, color="C0", alpha=0.8)
axes[1].set_xlabel(r"True x position $[\mathrm{m}]$", fontsize=plot_label_size)
axes[1].set_ylabel("Number of events", fontsize=plot_label_size)
axes[1].set_title("Training target x", fontsize=plot_title_size)
axes[1].tick_params(labelsize=plot_tick_size)

axes[2].hist(train_ypos, bins=50, color="C1", alpha=0.8)
axes[2].set_xlabel(r"True y position $[\mathrm{m}]$", fontsize=plot_label_size)
axes[2].set_ylabel("Number of events", fontsize=plot_label_size)
axes[2].set_title("Training target y", fontsize=plot_title_size)
axes[2].tick_params(labelsize=plot_tick_size)

density = axes[3].hist2d(
    train_xpos,
    train_ypos,
    bins=50,
    cmap="viridis",
)
axes[3].set_xlabel(r"True x position $[\mathrm{m}]$", fontsize=plot_label_size)
axes[3].set_ylabel(r"True y position $[\mathrm{m}]$", fontsize=plot_label_size)
axes[3].set_title("Training target density", fontsize=plot_title_size)
axes[3].tick_params(labelsize=plot_tick_size)
colorbar = fig.colorbar(density[3], ax=axes[3], pad=0.03)
colorbar.set_label("Number of events", fontsize=plot_label_size)
colorbar.ax.tick_params(labelsize=plot_tick_size)

fig.subplots_adjust(
    left=0.10,
    right=0.92,
    bottom=0.09,
    top=0.93,
    wspace=0.38,
    hspace=0.55,
)
fig.savefig(fig_dir / "dataset_overview.pdf", bbox_inches="tight")
None

### 6. Normalize hit features and targets

Neural networks usually train more reliably when inputs and targets have similar numerical scales. The normalization constants are computed from the training split only, then reused for validation and test data. This avoids leaking information from the validation or test split into the training procedure.

In [ ]:
feature_means = np.array([
    ak.mean(train_dataset["data"][:, 0, :]),
    ak.mean(train_dataset["data"][:, 1, :]),
    ak.mean(train_dataset["data"][:, 2, :]),
])

feature_stds = np.array([
    ak.std(train_dataset["data"][:, 0, :]),
    ak.std(train_dataset["data"][:, 1, :]),
    ak.std(train_dataset["data"][:, 2, :]),
])

target_means = np.array([
    ak.mean(train_dataset["xpos"]),
    ak.mean(train_dataset["ypos"]),
])

target_stds = np.array([
    ak.std(train_dataset["xpos"]),
    ak.std(train_dataset["ypos"]),
])


def normalize_dataset(dataset):
    times = dataset["data"][:, 0:1, :]
    hit_x = dataset["data"][:, 1:2, :]
    hit_y = dataset["data"][:, 2:3, :]

    normalized_times = (times - feature_means[0]) / feature_stds[0]
    normalized_hit_x = (hit_x - feature_means[1]) / feature_stds[1]
    normalized_hit_y = (hit_y - feature_means[2]) / feature_stds[2]

    dataset["data"] = ak.concatenate(
        [normalized_times, normalized_hit_x, normalized_hit_y],
        axis=1,
    )

    dataset["xpos"] = (dataset["xpos"] - target_means[0]) / target_stds[0]
    dataset["ypos"] = (dataset["ypos"] - target_means[1]) / target_stds[1]

    return dataset


train_dataset = normalize_dataset(train_dataset)
val_dataset = normalize_dataset(val_dataset)
test_dataset = normalize_dataset(test_dataset)

print("feature means [time, hit_x, hit_y]:", feature_means)
print("feature stds [time, hit_x, hit_y]:", feature_stds)
print("target means [xpos, ypos]:", target_means)
print("target stds [xpos, ypos]:", target_stds)

### 7. Custom batching for variable-length hit sequences

A normal PyTorch `DataLoader` tries to stack samples directly into one rectangular tensor. That does not work here because each event can have a different number of hits. The custom `collate_fn` keeps the hit sequences and their lengths together so the Transformer can pad and mask them later.

In [ ]:
def collate_fn_transformer(batch):
    data_list = []
    labels = []
    lengths = []

    for event in batch:
        tensor_data = torch.from_numpy(event["data"].to_numpy()).T
        tensor_data = tensor_data.to(dtype=torch.float32)

        data_list.append(tensor_data)
        lengths.append(tensor_data.shape[0])

        label = torch.tensor(
            [event["xpos"], event["ypos"]],
            dtype=torch.float32,
        )
        labels.append(label.unsqueeze(0))

    flat_hits = torch.cat(data_list, dim=0)
    labels = torch.cat(labels, dim=0)

    return [flat_hits, lengths], labels

In [ ]:
generator = torch.Generator().manual_seed(seed)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn_transformer,
    generator=generator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn_transformer,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn_transformer,
)

batch_data, batch_labels = next(iter(train_loader))
flat_hits, lengths = batch_data

print("flat hit tensor shape:", flat_hits.shape)
print("number of events in batch:", len(lengths))
print("first five sequence lengths:", lengths[:5])
print("label tensor shape:", batch_labels.shape)

### 8. Padding and masking

The Transformer encoder expects a batch tensor with shape `[batch_size, sequence_length, hidden_dim]`. Since the IceCube events have different numbers of hits, each batch is padded up to the longest event in that batch.

The padding values are artificial and should not influence attention or the final event representation. A boolean padding mask marks padded positions with `True`, so `nn.TransformerEncoder` can ignore them. After the encoder, masked mean pooling averages only over real hits and ignores padded tokens.

In [ ]:
class IceCubeTransformerEncoder(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        num_attention_heads,
        num_encoder_layers,
        dim_feedforward,
        dropout,
    ):
        super().__init__()

        self.input_projection = nn.Linear(input_dim, hidden_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_attention_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="relu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers,
        )

        self.output_head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, data):
        flat_hits, lengths = data

        embedded_hits = self.input_projection(flat_hits)
        event_parts = embedded_hits.split(lengths, dim=0)
        padded_events = pad_sequence(event_parts, batch_first=True)

        batch_size, max_length, _ = padded_events.shape
        padding_mask = torch.zeros(
            batch_size,
            max_length,
            dtype=torch.bool,
            device=padded_events.device,
        )

        for event_idx, length in enumerate(lengths):
            padding_mask[event_idx, length:] = True

        encoded_events = self.encoder(
            padded_events,
            src_key_padding_mask=padding_mask,
        )

        valid_mask = ~padding_mask
        summed_events = (encoded_events * valid_mask.unsqueeze(-1)).sum(dim=1)
        lengths_tensor = torch.tensor(lengths, device=encoded_events.device).unsqueeze(1)
        pooled_events = summed_events / lengths_tensor

        return self.output_head(pooled_events)

In [ ]:
model = IceCubeTransformerEncoder(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim,
    num_attention_heads=num_attention_heads,
    num_encoder_layers=num_encoder_layers,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
).to(device)

num_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

batch_data, batch_labels = next(iter(train_loader))
batch_data = [batch_data[0].to(device), batch_data[1]]

with torch.no_grad():
    batch_predictions = model(batch_data)

print(model)
print("trainable parameters:", num_parameters)
print("prediction shape:", batch_predictions.shape)
print("label shape:", batch_labels.shape)

### 9. Loss function and optimizer

This is a regression problem because the model predicts two continuous coordinates. We train with mean squared error in normalized coordinate space and use AdamW to update the Transformer parameters.

In [ ]:
loss_fn = nn.MSELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)

print(loss_fn)
print(optimizer)

### 10. Training and validation helpers

The training loop is written explicitly because each batch contains both the concatenated hit tensor and the sequence lengths. For every batch, the hit tensor and labels are moved to the selected device, the model predicts normalized coordinates, the MSE loss is backpropagated, and the optimizer updates the model parameters.

In [ ]:
def move_batch_to_device(batch_data, batch_labels):
    flat_hits, lengths = batch_data
    flat_hits = flat_hits.to(device)
    batch_labels = batch_labels.to(device)
    return [flat_hits, lengths], batch_labels


def train_one_epoch(model, data_loader, optimizer, loss_fn):
    model.train()
    batch_losses = []

    progress_bar = tqdm(data_loader, leave=False)

    for batch_data, batch_labels in progress_bar:
        batch_data, batch_labels = move_batch_to_device(batch_data, batch_labels)

        predictions = model(batch_data)
        loss = loss_fn(predictions, batch_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_value = loss.item()
        batch_losses.append(loss_value)
        progress_bar.set_postfix(loss=f"{loss_value:.4f}")

    return float(np.mean(batch_losses))


@torch.no_grad()
def evaluate_loss(model, data_loader, loss_fn):
    model.eval()
    batch_losses = []

    for batch_data, batch_labels in data_loader:
        batch_data, batch_labels = move_batch_to_device(batch_data, batch_labels)

        predictions = model(batch_data)
        loss = loss_fn(predictions, batch_labels)
        batch_losses.append(loss.item())

    return float(np.mean(batch_losses))

### 11. Train or load the Transformer

The best model is selected by validation loss and saved as a checkpoint. After training once, `skip_training` can be set to `True` to reload the saved model instead of running the training loop again.

In [ ]:
train_losses = []
val_losses = []

if skip_training and checkpoint_path.exists():
    checkpoint = load_checkpoint(
        checkpoint_path,
        model=model,
        optimizer=optimizer,
        device=device,
    )

    train_losses = checkpoint["train_losses"]
    val_losses = checkpoint["val_losses"]

    print("Loaded checkpoint from:", checkpoint_path)
    print("Best validation loss:", checkpoint["best_val_loss"])
    print("Best epoch:", checkpoint["epoch"])

else:
    best_val_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(num_epochs):
        train_loss = train_one_epoch(
            model=model,
            data_loader=train_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
        )

        val_loss = evaluate_loss(
            model=model,
            data_loader=val_loader,
            loss_fn=loss_fn,
        )

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        improved = val_loss < best_val_loss

        if improved:
            best_val_loss = val_loss
            epochs_without_improvement = 0

            save_checkpoint(
                path=checkpoint_path,
                model=model,
                optimizer=optimizer,
                epoch=epoch + 1,
                train_losses=train_losses,
                val_losses=val_losses,
                best_val_loss=best_val_loss,
                extra={
                    "feature_means": feature_means,
                    "feature_stds": feature_stds,
                    "target_means": target_means,
                    "target_stds": target_stds,
                },
            )
        else:
            epochs_without_improvement += 1

        print(
            f"Epoch {epoch + 1:02d}/{num_epochs} | "
            f"train loss: {train_loss:.4f} | "
            f"val loss: {val_loss:.4f} | "
            f"best val: {best_val_loss:.4f}"
        )

        if epochs_without_improvement >= patience:
            print(f"Early stopping after {epoch + 1} epochs.")
            break

    checkpoint = load_checkpoint(
        checkpoint_path,
        model=model,
        optimizer=optimizer,
        device=device,
    )

    print("Loaded best checkpoint from epoch:", checkpoint["epoch"])

### 12. Training and validation loss curves

The loss curves show whether the Transformer learned from the training data and whether validation performance improved at the same time. The best checkpoint epoch is marked so it is clear which model is used for evaluation.

In [ ]:
if len(train_losses) == 0 or len(val_losses) == 0:
    print("No losses available. Train the model or load a checkpoint first.")
else:
    epochs = np.arange(1, len(train_losses) + 1)

    fig, ax = plt.subplots(figsize=(6.8, 4.6))

    ax.plot(epochs, train_losses, label="Train Loss", color="C0", linewidth=2.2)
    ax.plot(epochs, val_losses, label="Validation Loss", color="C1", linewidth=2.2)

    ax.axvline(
        checkpoint["epoch"],
        color="black",
        linestyle=":",
        linewidth=1.8,
        label=f"Best checkpoint epoch: {checkpoint['epoch']}",
    )

    ax.set_xlabel("Epoch", fontsize=plot_label_size)
    ax.set_ylabel("MSE loss", fontsize=plot_label_size)
    ax.set_title("Transformer Training and Validation Loss", fontsize=plot_title_size)
    ax.tick_params(labelsize=plot_tick_size)
    ax.legend(fontsize=plot_legend_size)

    fig.tight_layout()
    fig.savefig(fig_dir / "transformer_loss_curves.pdf", bbox_inches="tight")

None

### 13. Test-set evaluation

The test set is used only after the model has been selected by validation loss. Predictions and labels are converted back from normalized coordinates to meters before computing the final reconstruction metrics.

In [ ]:
@torch.no_grad()
def collect_predictions(model, data_loader):
    model.eval()
    all_predictions = []
    all_labels = []

    for batch_data, batch_labels in data_loader:
        batch_data, batch_labels = move_batch_to_device(batch_data, batch_labels)
        predictions = model(batch_data)

        all_predictions.append(predictions.cpu())
        all_labels.append(batch_labels.cpu())

    all_predictions = torch.cat(all_predictions, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    return all_predictions, all_labels


test_loss = evaluate_loss(
    model=model,
    data_loader=test_loader,
    loss_fn=loss_fn,
)

all_preds, all_labels = collect_predictions(model, test_loader)

target_means_tensor = torch.tensor(target_means, dtype=torch.float32)
target_stds_tensor = torch.tensor(target_stds, dtype=torch.float32)

pred_positions = all_preds * target_stds_tensor + target_means_tensor
true_positions = all_labels * target_stds_tensor + target_means_tensor

error_xy = pred_positions - true_positions
position_error = torch.sqrt((error_xy ** 2).sum(dim=1))

mae_x = mean_absolute_error(true_positions[:, 0], pred_positions[:, 0])
mae_y = mean_absolute_error(true_positions[:, 1], pred_positions[:, 1])
mse_physical = mean_squared_error(true_positions, pred_positions)
r2 = r2_score(true_positions, pred_positions)

mean_position_error = position_error.mean().item()
median_position_error = position_error.median().item()

print(f"Test MSE loss in normalized space: {test_loss:.4f}")
print(f"Physical-space MSE: {mse_physical:.4f}")
print(f"MAE x: {mae_x:.4f} m")
print(f"MAE y: {mae_y:.4f} m")
print(f"Mean position error: {mean_position_error:.4f} m")
print(f"Median position error: {median_position_error:.4f} m")
print(f"R2 score: {r2:.4f}")

### 14. Reconstruction plots

The following plots compare predicted and true coordinates and show the distribution of two-dimensional reconstruction errors. These views make the regression quality easier to interpret than a single scalar metric.

The next plot compares predicted and true coordinates separately for x and y. Points close to the diagonal line correspond to accurate predictions.

In [ ]:
true_x = true_positions[:, 0].numpy()
true_y = true_positions[:, 1].numpy()
pred_x = pred_positions[:, 0].numpy()
pred_y = pred_positions[:, 1].numpy()

fig, axes = plt.subplots(1, 2, figsize=(8.8, 4.2))

plot_items = [
    (axes[0], true_x, pred_x, "C0", r"True x position $[\mathrm{m}]$", r"Predicted x position $[\mathrm{m}]$", "Predicted vs True x Position"),
    (axes[1], true_y, pred_y, "C1", r"True y position $[\mathrm{m}]$", r"Predicted y position $[\mathrm{m}]$", "Predicted vs True y Position"),
]

for ax, true_values, pred_values, color, xlabel, ylabel, title in plot_items:
    ax.scatter(
        true_values,
        pred_values,
        s=7,
        alpha=0.4,
        color=color,
        label="Test events",
    )

    min_value = min(true_values.min(), pred_values.min())
    max_value = max(true_values.max(), pred_values.max())

    ax.plot(
        [min_value, max_value],
        [min_value, max_value],
        color="black",
        linestyle=":",
        linewidth=1.8,
        label="Perfect prediction",
    )

    ax.set_xlabel(xlabel, fontsize=plot_label_size)
    ax.set_ylabel(ylabel, fontsize=plot_label_size)
    ax.set_title(title, fontsize=plot_title_size)
    ax.tick_params(labelsize=plot_tick_size)
    ax.legend(fontsize=plot_legend_size)

fig.tight_layout()
fig.savefig(fig_dir / "predicted_vs_true_positions.pdf", bbox_inches="tight")
None

This histogram shows the two-dimensional reconstruction error in meters. The vertical lines summarize the mean and median error over the test set.

In [ ]:
position_error_np = position_error.numpy()

position_error_mean = position_error.mean().item()
position_error_median = position_error.median().item()
position_error_std = position_error.std().item()
position_error_sem = position_error_std / np.sqrt(len(position_error_np))

fig, ax = plt.subplots(figsize=(6.8, 4.6))

ax.hist(
    position_error_np,
    bins=50,
    color="C0",
    alpha=0.8,
)

ax.axvline(
    position_error_mean,
    color="C1",
    linestyle="--",
    label=rf"Mean: ${position_error_mean:.3f}\,\mathrm{{m}}$",
)

ax.axvline(
    position_error_median,
    color="C3",
    linestyle=":",
    label=rf"Median: ${position_error_median:.3f}\,\mathrm{{m}}$",
)

ax.plot([], [], linestyle="None", label=rf"Std: ${position_error_std:.3f}\,\mathrm{{m}}$")
ax.plot([], [], linestyle="None", label=rf"SEM: ${position_error_sem:.3f}\,\mathrm{{m}}$")

ax.set_xlabel(r"Position error $[\mathrm{m}]$", fontsize=plot_label_size)
ax.set_ylabel("Number of events", fontsize=plot_label_size)
ax.set_title("Distribution of Position Reconstruction Error", fontsize=plot_title_size)
ax.tick_params(labelsize=plot_tick_size)

handles, labels = ax.get_legend_handles_labels()
order = [0, 2, 3, 1]
ax.legend(
    [handles[i] for i in order],
    [labels[i] for i in order],
    fontsize=plot_legend_size,
)

fig.tight_layout()
fig.savefig(fig_dir / "position_error_histogram.pdf", bbox_inches="tight")
None

This 2D view shows a random subset of true and reconstructed event positions. Each line connects one true position to the corresponding prediction, so shorter lines mean smaller reconstruction errors.

In [ ]:
rng = np.random.default_rng(seed)
num_events_to_plot = min(300, len(true_x))
plot_indices = rng.choice(len(true_x), size=num_events_to_plot, replace=False)

fig, ax = plt.subplots(figsize=(6.4, 6.0))

for idx in plot_indices:
    ax.plot(
        [true_x[idx], pred_x[idx]],
        [true_y[idx], pred_y[idx]],
        color="black",
        alpha=0.15,
        linewidth=0.6,
    )

ax.scatter(
    true_x[plot_indices],
    true_y[plot_indices],
    s=14,
    color="C0",
    alpha=0.8,
    label="True positions",
)

ax.scatter(
    pred_x[plot_indices],
    pred_y[plot_indices],
    s=14,
    color="C3",
    alpha=0.8,
    label="Predicted positions",
)

ax.set_xlabel(r"x position $[\mathrm{m}]$", fontsize=plot_label_size)
ax.set_ylabel(r"y position $[\mathrm{m}]$", fontsize=plot_label_size)
ax.set_title("True and Reconstructed Event Positions", fontsize=plot_title_size)
ax.set_aspect("equal", adjustable="box")
ax.tick_params(labelsize=plot_tick_size)
ax.legend(fontsize=plot_legend_size)

fig.tight_layout()
fig.savefig(fig_dir / "true_vs_reconstructed_positions_2d.pdf", bbox_inches="tight")
None

The final diagnostic checks whether reconstruction error depends on the number of hits in an event. The binned trend helps reveal whether events with more detected photons are generally easier to reconstruct.

In [ ]:
test_hit_counts = ak.to_numpy(ak.num(test_dataset["data"][:, 0, :], axis=1))

num_bins = 12
bin_edges = np.quantile(test_hit_counts, np.linspace(0, 1, num_bins + 1))
bin_edges = np.unique(bin_edges)

bin_centers = []
mean_errors = []
std_errors = []

for left_edge, right_edge in zip(bin_edges[:-1], bin_edges[1:]):
    in_bin = (test_hit_counts >= left_edge) & (test_hit_counts <= right_edge)
    errors_in_bin = position_error_np[in_bin]

    if len(errors_in_bin) == 0:
        continue

    bin_centers.append(0.5 * (left_edge + right_edge))
    mean_errors.append(errors_in_bin.mean())
    std_errors.append(errors_in_bin.std())

fig, ax = plt.subplots(figsize=(6.8, 4.6))

ax.scatter(
    test_hit_counts,
    position_error_np,
    s=6,
    alpha=0.25,
    color="C0",
    label="Test events",
)

ax.errorbar(
    bin_centers,
    mean_errors,
    yerr=std_errors,
    color="C3",
    marker="o",
    linewidth=2.2,
    capsize=3,
    label="Binned mean $\pm$ std",
)

ax.set_xlabel("Number of hits", fontsize=plot_label_size)
ax.set_ylabel(r"Position error $[\mathrm{m}]$", fontsize=plot_label_size)
ax.set_title("Position Error vs Number of Hits", fontsize=plot_title_size)
ax.tick_params(labelsize=plot_tick_size)
ax.legend(fontsize=plot_legend_size)

fig.tight_layout()
fig.savefig(fig_dir / "position_error_vs_hit_count.pdf", bbox_inches="tight")
None